# 05. Sessions

**Session**은 여러 번의 `Runner.run` 호출에 걸쳐 **대화 히스토리를 자동으로 유지**하는 기능입니다.
- 세션을 쓰지 않으면 매 턴마다 이전 메시지 목록을 직접 구성해 넘겨야 합니다.
- `SQLiteSession(session_id)`를 사용하면 SDK가 대화를 DB에 저장/불러와서, 같은 `session`으로 실행할 때마다 맥락이 이어집니다.

활용: 채팅봇, 고객지원 에이전트 등 다회차 대화.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from agents import Agent, Runner, SQLiteSession

Model = "gpt-5-nano"

## 세션 없이 실행 (맥락 없음)

세션 없이 두 번 호출하면, 두 번째 질문 "What state is it in?"에 대한 맥락(금문교가 샌프란시스코에 있다는 것)이 없습니다.

In [4]:
agent = Agent(
    name="Assistant",
    instructions="간결하게 답변해 주세요.",
    model=Model
)

In [7]:
question_1 = "남산 팔각정은 어느 도시에 있나요?"
result1 = await Runner.run(agent, question_1)
print("Q1:", question_1)
print("A1:", result1.final_output)

Q1: 남산 팔각정은 어느 도시에 있나요?
A1: 서울특별시, 대한민국에 있습니다.


In [9]:
question_2 = "그 도시는 어느 나라에 있나요?"
result2 = await Runner.run(agent, question_2)
print("Q2:", question_2)
print("A2:", result2.final_output)

Q2: 그 도시는 어느 나라에 있나요?
A2: 어떤 도시인지 도시 이름을 알려주시면, 그 도시가 어느 나라에 있는지 알려드리겠습니다. 필요하시면 추가 정보도 확인해 드릴게요.


## SQLiteSession으로 대화 이어가기

같은 `session` 인스턴스를 넘기면, 이전 턴의 메시지가 자동으로 포함됩니다.

In [10]:
session = SQLiteSession("conversation_123")

In [11]:
question_1 = "남산 팔각정은 어느 도시에 있나요?"
result1 = await Runner.run(
    agent, 
    question_1,
    session=session,
)
print("Q1:", question_1)
print("A1:", result1.final_output)

Q1: 남산 팔각정은 어느 도시에 있나요?
A1: 서울특별시에 있습니다. 남산(서울 중심부)에 위치한 팔각정입니다.


In [12]:
question_2 = "그 도시는 어느 나라에 있나요?"
result2 = await Runner.run(
    agent, 
    question_2,
    session=session
)
print("Q2:", question_2)
print("A2:", result2.final_output)

Q2: 그 도시는 어느 나라에 있나요?
A2: 대한민국(한국)에 있습니다.


In [13]:
question_3 = "그곳 인구는 얼마인가요?"
result3 = await Runner.run(
    agent, 
    question_3,
    session=session
)
print("Q3:", question_3)
print("A3:", result3.final_output)

Q3: 그곳 인구는 얼마인가요?
A3: 서울특별시의 인구는 약 9.6백만 명(약 9,600,000명) 정도입니다. 정확한 최신 수치는 통계청이나 서울시 자료를 확인해 주세요.


## 세션 ID별로 대화 분리

`session_id`가 다르면 서로 다른 대화로 저장됩니다.
예: `SQLiteSession("user_456", "conversations.db")`로 사용자/채널별 분리.

In [15]:
session_b = SQLiteSession("another_user_456")
result_b = await Runner.run(agent, "남산 팔각정은 어느 도시에 있나요?", session=session_b)
print("User B - A1:", result_b.final_output)

User B - A1: 서울특별시(대한민국)입니다.


## 정리

- `SQLiteSession(session_id)` 또는 `SQLiteSession(session_id, "conversations.db")`: 파일 DB에 대화 저장
- `Runner.run_sync(agent, input, session=session)`: 같은 session을 넘기면 히스토리 자동 유지
- 세션 ID 전략: 사용자ID, 채널, 날짜 등을 조합해 대화 단위 구분